# Y-Net Segmentation × Classification Agreement Experiment

Run the trained Y-Net backbone (segmentation) together with an externally
trained classifier artifact (Logistic Regression / Linear SVM / Ridge /
Single ELM / Ensemble ELM) on a random sample of N images from any dataset
(Kvasir-SEG, EDD2020, PolypGen, ...), and bucket each sample into one of:

1. Correct Segmentation + Correct Classification
2. Correct Segmentation + Incorrect Classification
3. Incorrect Segmentation + Correct Classification
4. Incorrect Segmentation + Incorrect Classification

**How to use:** run every cell top to bottom once, then just edit the paths
in the **"⚙️ Configuration"** cell near the bottom and re-run that cell plus
the final "Run Experiment" cell for each new dataset — no need to touch
anything above it.


## 1. Imports

In [1]:
import os
import glob
import random
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
import joblib
from tensorflow.keras import layers


I0000 00:00:1788754397.624100   17095 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 2. Custom Keras objects (must match training-time definitions exactly)

In [3]:
@tf.keras.utils.register_keras_serializable()
class KerasPCCLayer(layers.Layer):
    def __init__(self, **kwargs):
        super(KerasPCCLayer, self).__init__(**kwargs)

    def call(self, inputs):
        epsilon = 1e-8
        mean = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        std = tf.math.reduce_std(inputs, axis=-1, keepdims=True) + epsilon
        norm_inputs = (inputs - mean) / std
        pcc_matrix = tf.matmul(tf.expand_dims(norm_inputs, -1), tf.expand_dims(norm_inputs, 1))
        return layers.Flatten()(pcc_matrix)


def dice_coef(y_true, y_pred, smooth=1.0):
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)


def bce_dice_loss(y_true, y_pred):
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    return bce + dice_loss(y_true, y_pred)


def focal_tversky_loss(y_true, y_pred, alpha=0.7, beta=0.3, gamma=0.75):
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    tp = tf.reduce_sum(y_true_f * y_pred_f)
    fn = tf.reduce_sum(y_true_f * (1.0 - y_pred_f))
    fp = tf.reduce_sum((1.0 - y_true_f) * y_pred_f)
    smooth = 1e-6
    tversky_index = (tp + smooth) / (tp + alpha * fn + beta * fp + smooth)
    return tf.math.pow((1.0 - tversky_index), gamma)


def iou_metric(y_true, y_pred):
    y_true_f = tf.cast(y_true, tf.float32)
    y_pred_f = tf.cast(y_pred > 0.5, tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + 1.0) / (union + 1.0)


CUSTOM_OBJECTS = {
    'KerasPCCLayer': KerasPCCLayer,
    'dice_coef': dice_coef,
    'iou_metric': iou_metric,
    'bce_dice_loss': bce_dice_loss,
    'focal_tversky_loss': focal_tversky_loss,
}


## 3. Universal classifier artifact loader

Auto-detects and loads whichever classifier artifact you point it at:
Logistic Regression / Single ELM / Ensemble ELM (`.npz`), or Linear SVM /
Ridge (`.joblib`).

In [4]:
class ClassifierArtifact:
    def __init__(self, kind, **kw):
        self.kind = kind
        self.__dict__.update(kw)

    def predict_proba(self, feats_flat_1024):
        """feats_flat_1024: raw (1024,) feature vector straight from the
        target Keras layer (e.g. clf_pcc_layer). Returns a single float
        probability in [0, 1] for the positive (Polyp) class."""
        x = feats_flat_1024
        if getattr(self, "selector_support", None) is not None:
            x = x[self.selector_support]

        if self.kind in ("svm", "ridge"):
            x_scaled = (x - self.scaler_center) / self.scaler_scale
            prob = self.sk_model.predict_proba(x_scaled.reshape(1, -1))[:, 1]
            return float(prob[0])

        x_scaled = (x - self.scaler_center) / self.scaler_scale

        if self.kind == "lr":
            logit = np.dot(x_scaled, self.coef.T) + self.intercept
            prob = 1.0 / (1.0 + np.exp(-logit))
            return float(np.asarray(prob).flatten()[0])

        if self.kind == "single_elm":
            h = np.tanh(np.dot(x_scaled.reshape(1, -1), self.W_input) + self.b_input)
            raw = np.dot(h, self.W_output)
            if self.val_min_score is None or self.val_max_score is None:
                prob = 1.0 / (1.0 + np.exp(-raw))  # fallback: logistic squashing (no saved calibration bounds)
            else:
                prob = (raw - self.val_min_score) / (self.val_max_score - self.val_min_score + 1e-7)
            return float(np.asarray(prob).flatten()[0])

        if self.kind == "ensemble_elm":
            h = np.tanh(np.dot(x_scaled.reshape(1, -1), self.W_input) + self.b_input)
            all_raw = [np.dot(h, w) for w in self.ensemble_weights]
            raw = np.mean(all_raw, axis=0)
            if self.val_min_score is None or self.val_max_score is None:
                prob = 1.0 / (1.0 + np.exp(-raw))  # fallback: logistic squashing (no saved calibration bounds)
            else:
                prob = (raw - self.val_min_score) / (self.val_max_score - self.val_min_score + 1e-7)
            return float(np.asarray(prob).flatten()[0])

        raise ValueError(f"Unknown classifier kind: {self.kind}")


def _get_key(d, *names):
    """Looks up the first matching key among several possible spellings
    (handles the older elm_artifacts.npz convention which used a trailing
    underscore, e.g. 'scaler_center_' instead of 'scaler_center')."""
    for n in names:
        if n in d.files:
            return d[n]
    return None


def load_classifier_artifact(path):
    if path.endswith(".joblib"):
        d = joblib.load(path)
        if "calibrated_svm" in d:
            return ClassifierArtifact(
                "svm", sk_model=d["calibrated_svm"], selector_support=d["selector_support"],
                scaler_center=d["scaler_center"], scaler_scale=d["scaler_scale"],
                threshold=float(d["decision_threshold"]), target_layer_name=d["target_layer_name"],
                img_size=int(d["img_size"]),
            )
        if "calibrated_ridge" in d:
            return ClassifierArtifact(
                "ridge", sk_model=d["calibrated_ridge"], selector_support=d["selector_support"],
                scaler_center=d["scaler_center"], scaler_scale=d["scaler_scale"],
                threshold=float(d["decision_threshold"]), target_layer_name=d["target_layer_name"],
                img_size=int(d["img_size"]),
            )
        raise ValueError(f"Unrecognized .joblib artifact schema: {list(d.keys())}")

    if path.endswith(".npz"):
        d = np.load(path, allow_pickle=True)
        keys = d.files

        selector_support = _get_key(d, "selector_support")
        scaler_center = _get_key(d, "scaler_center", "scaler_center_")
        scaler_scale = _get_key(d, "scaler_scale", "scaler_scale_")
        if scaler_center is None or scaler_scale is None:
            raise KeyError(
                f"Could not find scaler center/scale in this artifact. "
                f"Available keys: {keys}. Expected one of "
                f"'scaler_center'/'scaler_center_' and 'scaler_scale'/'scaler_scale_'."
            )

        threshold_raw = _get_key(d, "decision_threshold")
        target_layer_raw = _get_key(d, "target_layer_name")
        img_size_raw = _get_key(d, "img_size")

        threshold = float(threshold_raw) if threshold_raw is not None else 0.5
        target_layer_name = str(target_layer_raw) if target_layer_raw is not None else "clf_pcc_layer"
        img_size = int(img_size_raw) if img_size_raw is not None else 256

        missing_meta = [n for n, v in [("decision_threshold", threshold_raw),
                                        ("target_layer_name", target_layer_raw),
                                        ("img_size", img_size_raw)] if v is None]
        if missing_meta:
            print(f"⚠️ Artifact is missing {missing_meta} — using defaults "
                  f"(threshold=0.5, target_layer_name='clf_pcc_layer', img_size=256). "
                  f"If your feature layer or resolution differs, pass them in manually.")

        common = dict(
            selector_support=selector_support, scaler_center=scaler_center, scaler_scale=scaler_scale,
            threshold=threshold, target_layer_name=target_layer_name, img_size=img_size,
        )

        if "coef" in keys:
            return ClassifierArtifact("lr", coef=d["coef"], intercept=d["intercept"], **common)

        if "ensemble_weights" in keys:
            val_min = _get_key(d, "val_min_score")
            val_max = _get_key(d, "val_max_score")
            if val_min is None or val_max is None:
                print("⚠️ Artifact has no saved val_min_score/val_max_score — falling back to logistic "
                      "squashing of the raw ensemble score instead of the original min-max rescale. "
                      "Threshold defaults to 0.5 (raw score 0) unless overridden.")
            return ClassifierArtifact(
                "ensemble_elm", W_input=d["W_input"], b_input=d["b_input"],
                ensemble_weights=list(d["ensemble_weights"]),
                val_min_score=float(val_min) if val_min is not None else None,
                val_max_score=float(val_max) if val_max is not None else None,
                **common,
            )

        if "W_output" in keys:
            val_min = _get_key(d, "val_min_score")
            val_max = _get_key(d, "val_max_score")
            if val_min is None or val_max is None:
                print("⚠️ Artifact has no saved val_min_score/val_max_score — falling back to logistic "
                      "squashing of the raw score instead of the original min-max rescale.")
            return ClassifierArtifact(
                "single_elm", W_input=d["W_input"], b_input=d["b_input"], W_output=d["W_output"],
                val_min_score=float(val_min) if val_min is not None else None,
                val_max_score=float(val_max) if val_max is not None else None,
                **common,
            )

        raise ValueError(f"Unrecognized .npz artifact schema: {keys}")

    raise ValueError(f"Unsupported classifier artifact extension: {path}")


## 4. Preprocessing & mask matching helpers

In [5]:
def preprocess_image(img_bgr, img_size):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_res = cv2.resize(img_rgb, (img_size, img_size))

    lab = cv2.cvtColor(img_res, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l_channel)
    limg = cv2.merge((cl, a_channel, b_channel))
    enhanced_img = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

    median_blur = cv2.medianBlur(enhanced_img, 3)
    sharpened = cv2.addWeighted(enhanced_img, 1.5, median_blur, -0.5, 0)
    return (sharpened / 255.0).astype(np.float32)


def load_mask(mask_path, img_size):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (img_size, img_size))
    mask_bin = (mask > 127).astype(np.float32)
    return mask_bin


def find_mask_for_image(img_path, masks_dir, mask_suffix=""):
    """Tries a few common naming conventions used across polyp datasets."""
    filename = os.path.basename(img_path)
    name, ext = os.path.splitext(filename)

    candidates = []
    if mask_suffix:
        candidates.append(os.path.join(masks_dir, f"{name}{mask_suffix}{ext}"))
        candidates.append(os.path.join(masks_dir, f"{name}{mask_suffix}.png"))
    candidates.append(os.path.join(masks_dir, filename))              # same filename (Kvasir-SEG style)
    candidates.append(os.path.join(masks_dir, f"{name}_mask{ext}"))   # PolypGen style
    candidates.append(os.path.join(masks_dir, f"{name}.png"))         # extension normalized to png

    for c in candidates:
        if os.path.exists(c):
            return c
    return None


def dice_score_np(pred_bin, gt_bin, smooth=1.0):
    intersection = np.sum(pred_bin * gt_bin)
    return (2.0 * intersection + smooth) / (np.sum(pred_bin) + np.sum(gt_bin) + smooth)


def iou_score_np(pred_bin, gt_bin, smooth=1.0):
    intersection = np.sum(pred_bin * gt_bin)
    union = np.sum(pred_bin) + np.sum(gt_bin) - intersection
    return (intersection + smooth) / (union + smooth)


## 5. Main experiment function (multi-class aware)

In [6]:
def gather_class_pairs(class_sources, mask_suffix_default=""):
    """
    class_sources: list of dicts, one per image class/folder, e.g.

        [
            {"name": "Polyp",        "images_dir": "...", "masks_dir": "...",  "label": 1},
            {"name": "Z-line",       "images_dir": "...", "masks_dir": None,   "label": 0},
            {"name": "Esophagitis",  "images_dir": "...", "masks_dir": None,   "label": 0},
        ]

    Classes with masks_dir=None get a virtual all-black mask (no ground-truth
    polyp region to segment) — this rewards the model for predicting an
    (almost) empty mask on non-polyp images, and penalizes false-positive
    segmentation, exactly like the VIRTUAL_BLACK_MASK convention used in your
    own training pipeline for negative sequences.

    Returns a list of tuples: (img_path, mask_path_or_None, label, class_name)
    """
    valid_exts = ("*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.PNG", "*.JPG", "*.JPEG")
    all_pairs = []

    for src in class_sources:
        name = src["name"]
        images_dir = src["images_dir"]
        masks_dir = src.get("masks_dir")
        label = src["label"]
        mask_suffix = src.get("mask_suffix", mask_suffix_default)

        images = []
        for ext in valid_exts:
            images.extend(glob.glob(os.path.join(images_dir, ext)))
        images = sorted(images)

        kept = 0
        for img_path in images:
            if masks_dir:
                mask_path = find_mask_for_image(img_path, masks_dir, mask_suffix)
                if mask_path is None:
                    continue  # this class has masks in general but this image's mask is missing -> skip
            else:
                mask_path = None  # virtual all-black mask will be used at load time
            all_pairs.append((img_path, mask_path, label, name))
            kept += 1

        print(f"  [{name}] found {len(images)} images, using {kept} (label={label}, masks={'yes' if masks_dir else 'virtual-empty'})")

    return all_pairs


def sample_pairs(all_pairs, num_samples=None, samples_per_class=None, seed=42):
    random.seed(seed)
    if samples_per_class:
        sampled = []
        for cls_name, n in samples_per_class.items():
            cls_pairs = [p for p in all_pairs if p[3] == cls_name]
            n_take = min(n, len(cls_pairs))
            sampled.extend(random.sample(cls_pairs, n_take))
        random.shuffle(sampled)
        return sampled
    n_take = min(num_samples, len(all_pairs))
    return random.sample(all_pairs, n_take)


def run_experiment(class_sources, dataset_name="Dataset", num_samples=100, samples_per_class=None,
                    img_size=256, seg_threshold=0.5, mask_suffix="",
                    output_dir="./agreement_experiment_results",
                    save_visuals=False, seed=42,
                    model_path=None, classifier_path=None,
                    _base_model=None, _clf=None):
    random.seed(seed)
    np.random.seed(seed)

    if _base_model is not None and _clf is not None:
        # Reuse an already-loaded model/classifier (handy when looping over
        # several datasets in the same notebook session without reloading).
        base_model, clf = _base_model, _clf
    else:
        print(f"🔄 Loading Y-Net backbone: {model_path}")
        base_model = tf.keras.models.load_model(model_path, custom_objects=CUSTOM_OBJECTS, compile=False)
        print(f"🔄 Loading classifier artifact: {classifier_path}")
        clf = load_classifier_artifact(classifier_path)
        print(f"   -> Detected classifier type: {clf.kind} | feature layer: {clf.target_layer_name} | threshold: {clf.threshold:.3f}")

    feature_extractor = tf.keras.Model(
        inputs=base_model.input,
        outputs=base_model.get_layer(clf.target_layer_name).output,
    )

    print("📊 Gathering image/mask pairs per class...")
    all_pairs = gather_class_pairs(class_sources, mask_suffix_default=mask_suffix)
    print(f"📊 Total pool: {len(all_pairs)} images across {len(class_sources)} classes.")
    if len(all_pairs) == 0:
        raise RuntimeError("No images found — check the paths in class_sources.")

    sampled_pairs = sample_pairs(all_pairs, num_samples=num_samples, samples_per_class=samples_per_class, seed=seed)
    print(f"🎲 Sampled {len(sampled_pairs)} images (seed={seed}).")

    CASE_NAMES = {
        1: "Correct Seg + Correct Clf",
        2: "Correct Seg + Incorrect Clf",
        3: "Incorrect Seg + Correct Clf",
        4: "Incorrect Seg + Incorrect Clf",
    }
    if save_visuals:
        for name in CASE_NAMES.values():
            os.makedirs(os.path.join(output_dir, "visuals", name.replace(" ", "_")), exist_ok=True)

    records = []
    for i, (img_path, mask_path, gt_label, class_name) in enumerate(sampled_pairs):
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            print(f"⚠️ Skipping unreadable image: {img_path}")
            continue

        preproc_img = preprocess_image(img_bgr, img_size)

        if mask_path is not None:
            gt_mask = load_mask(mask_path, img_size)
        else:
            gt_mask = np.zeros((img_size, img_size), dtype=np.float32)  # virtual empty mask

        inp = np.expand_dims(preproc_img, axis=0).astype(np.float32)

        outputs = base_model(inp, training=False)
        seg_pred = outputs[0].numpy()[0, :, :, 0] if isinstance(outputs, (list, tuple)) else outputs.numpy()[0, :, :, 0]
        seg_pred_bin = (seg_pred > 0.5).astype(np.float32)

        dice = dice_score_np(seg_pred_bin, gt_mask)
        iou = iou_score_np(seg_pred_bin, gt_mask)
        seg_correct = dice >= seg_threshold

        feats = feature_extractor(inp, training=False)
        feats_flat = tf.reshape(feats, [tf.shape(feats)[0], -1]).numpy().flatten().astype(np.float32)
        prob = clf.predict_proba(feats_flat)
        pred_label = int(prob > clf.threshold)

        clf_correct = pred_label == gt_label

        if seg_correct and clf_correct:
            case = 1
        elif seg_correct and not clf_correct:
            case = 2
        elif not seg_correct and clf_correct:
            case = 3
        else:
            case = 4

        records.append({
            "filename": os.path.basename(img_path), "class_name": class_name,
            "has_gt_mask": mask_path is not None,
            "dice": dice, "iou": iou, "seg_correct": seg_correct,
            "gt_label": gt_label, "pred_label": pred_label, "pred_prob": prob,
            "clf_correct": clf_correct, "case": case, "case_name": CASE_NAMES[case],
        })

        if save_visuals:
            overlay = (preproc_img * 255).astype(np.uint8).copy()
            gt_edge = cv2.Canny((gt_mask * 255).astype(np.uint8), 50, 150)
            pred_edge = cv2.Canny((seg_pred_bin * 255).astype(np.uint8), 50, 150)
            overlay[gt_edge > 0] = [0, 255, 0]
            overlay[pred_edge > 0] = [255, 0, 0]
            out_name = f"{class_name}_{os.path.basename(img_path)}"
            out_path = os.path.join(output_dir, "visuals", CASE_NAMES[case].replace(" ", "_"), out_name)
            cv2.imwrite(out_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

        if (i + 1) % 10 == 0 or (i + 1) == len(sampled_pairs):
            print(f"  ...processed {i + 1}/{len(sampled_pairs)}")

    df = pd.DataFrame(records)
    os.makedirs(output_dir, exist_ok=True)
    per_image_csv = os.path.join(output_dir, f"{dataset_name}_per_image_results.csv")
    df.to_csv(per_image_csv, index=False)

    summary_rows = []
    for case_id, case_name in CASE_NAMES.items():
        sub = df[df["case"] == case_id]
        count = len(sub)
        pct = 100.0 * count / len(df) if len(df) > 0 else 0.0
        summary_rows.append({
            "Case": case_name, "Count": count, "Percentage": f"{pct:.2f}%",
            "Mean Dice": f"{sub['dice'].mean():.4f}" if count else "-",
            "Mean IoU": f"{sub['iou'].mean():.4f}" if count else "-",
            "Mean Clf Prob": f"{sub['pred_prob'].mean():.4f}" if count else "-",
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_csv = os.path.join(output_dir, f"{dataset_name}_summary.csv")
    summary_df.to_csv(summary_csv, index=False)

    # Per-class x case cross-tab — useful to see e.g. which non-polyp class
    # (Z-line, Esophagitis, ...) drives most classification errors.
    crosstab = pd.crosstab(df["class_name"], df["case_name"])
    crosstab_csv = os.path.join(output_dir, f"{dataset_name}_class_case_crosstab.csv")
    crosstab.to_csv(crosstab_csv)

    print(f"\n📊 ===== {dataset_name} — AGREEMENT EXPERIMENT SUMMARY (N={len(df)}) =====")
    print(summary_df.to_string(index=False))
    print(f"\n📊 ===== Per-class breakdown =====")
    print(crosstab.to_string())
    print(f"\n💾 Per-image results saved to: {per_image_csv}")
    print(f"💾 Summary table saved to:     {summary_csv}")
    print(f"💾 Class x case crosstab saved to: {crosstab_csv}")
    if save_visuals:
        print(f"🖼️  Overlay visuals saved under: {os.path.join(output_dir, 'visuals')}")

    return df, summary_df, crosstab, base_model, clf


## 6. ⚙️ Configuration — edit these paths for each run

For **Kvasir-V2** (mixed polyp / non-polyp classes): give each class its own
image folder. Only the `Polyp` class needs a `masks_dir` — every other class
(Z-line, Esophagitis, Dyed-lifted-polyps that aren't actually polyps, etc.)
gets `masks_dir=None`, and the script automatically treats those as
`label=0` (Normal) with a virtual empty ground-truth mask (so a correct
segmentation there just means "predicted nothing", matching your own
`VIRTUAL_BLACK_MASK` convention from training).

For a single polyp-only dataset (like the earlier Kvasir-SEG run), just
provide one entry in the list.

In [23]:
MODEL_PATH = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/thesis_v3_sequential_v_eval_data_driven_v2.keras"
CLASSIFIER_PATH = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/elm_artifacts.npz"    # or logreg_artifacts.npz / single_elm_artifacts.npz / linear_svm_artifacts.joblib / ridge_classifier_artifacts.joblib

DATASET_NAME = "Kvasir-V2"

# One entry per class/folder. "masks_dir": None for classes with no ground-truth mask.
CLASS_SOURCES = [
    {"name": "Polyp",        "images_dir": "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/PolypGen2021_MultiCenterData_v2/sequenceData/positive/seq3/images_seq3",          "masks_dir": "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/PolypGen2021_MultiCenterData_v2/sequenceData/positive/seq3/masks_seq3", "label": 1},
    {"name": "Z-line",       "images_dir": "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/PolypGen2021_MultiCenterData_v2/negativeOnly/seq13_neg",          "masks_dir": None,                              "label": 0},
    {"name": "Esophagitis",  "images_dir": "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/PolypGen2021_MultiCenterData_v2/negativeOnly/seq10_neg",     "masks_dir": None,                              "label": 0},
    # add more classes here the same way, e.g. "Pylorus", "Cecum", "Ulcerative-colitis", ...
]

NUM_SAMPLES = 500          # used only if SAMPLES_PER_CLASS is None -> sampled from the pooled set across all classes
SAMPLES_PER_CLASS = None   # or set e.g. {"Polyp": 40, "Z-line": 30, "Esophagitis": 30} for a fixed split per class

SEG_THRESHOLD = 0.5        # Dice >= this value counts as "correct" segmentation
MASK_SUFFIX = ""           # default mask filename suffix, e.g. "_mask"; override per-class via "mask_suffix" key if needed
OUTPUT_DIR = f"./agreement_results_{DATASET_NAME}"
SAVE_VISUALS = True
SEED = 42


## 7. Run the experiment

In [10]:
df, summary_df, crosstab, _model, _clf = run_experiment(
    class_sources=CLASS_SOURCES,
    dataset_name=DATASET_NAME,
    num_samples=NUM_SAMPLES,
    samples_per_class=SAMPLES_PER_CLASS,
    seg_threshold=SEG_THRESHOLD,
    mask_suffix=MASK_SUFFIX,
    output_dir=OUTPUT_DIR,
    save_visuals=SAVE_VISUALS,
    seed=SEED,
    model_path=MODEL_PATH,
    classifier_path=CLASSIFIER_PATH,
)
summary_df


🔄 Loading Y-Net backbone: /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/thesis_v3_sequential_v_eval_data_driven_v2.keras
🔄 Loading classifier artifact: /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/elm_artifacts.npz
⚠️ Artifact is missing ['decision_threshold', 'target_layer_name', 'img_size'] — using defaults (threshold=0.5, target_layer_name='clf_pcc_layer', img_size=256). If your feature layer or resolution differs, pass them in manually.
⚠️ Artifact has no saved val_min_score/val_max_score — falling back to logistic squashing of the raw ensemble score instead of the original min-max rescale. Threshold defaults to 0.5 (raw score 0) unless overridden.
   -> Detected classifier type: ensemble_elm | feature layer: clf_pcc_layer | threshold: 0.500
📊 Gathering image/mask pairs per class...
  [Polyp] found 1000 images, using 1000 (label=1, masks=yes)
  [Z-line] found 1000 images, using 1000 (label=0, masks=virtual-empty)


I0000 00:00:1788754609.298252   17095 cuda_dnn.cc:461] Loaded cuDNN version 92101


  ...processed 10/100
  ...processed 20/100
  ...processed 30/100
  ...processed 40/100
  ...processed 50/100
  ...processed 60/100
  ...processed 70/100
  ...processed 80/100
  ...processed 90/100
  ...processed 100/100

📊 ===== Kvasir-V2 — AGREEMENT EXPERIMENT SUMMARY (N=100) =====
                         Case  Count Percentage Mean Dice Mean IoU Mean Clf Prob
    Correct Seg + Correct Clf     32     32.00%    0.8551   0.7719        0.6875
  Correct Seg + Incorrect Clf     12     12.00%    0.9779   0.9585        0.6053
  Incorrect Seg + Correct Clf     12     12.00%    0.0047   0.0044        0.3563
Incorrect Seg + Incorrect Clf     44     44.00%    0.0110   0.0097        0.6589

📊 ===== Per-class breakdown =====
case_name    Correct Seg + Correct Clf  Correct Seg + Incorrect Clf  Incorrect Seg + Correct Clf  Incorrect Seg + Incorrect Clf
class_name                                                                                                                     
Esophagitis        

,Case,Count,Percentage,Mean Dice,Mean IoU,Mean Clf Prob
0,Correct Seg + Correct Clf,32,32.00%,0.8551,0.7719,0.6875
1,Correct Seg + Incorrect Clf,12,12.00%,0.9779,0.9585,0.6053
2,Incorrect Seg + Correct Clf,12,12.00%,0.0047,0.0044,0.3563
3,Incorrect Seg + Incorrect Clf,44,44.00%,0.0110,0.0097,0.6589


## 8. (Optional) Inspect a few examples per case

Useful for pulling qualitative figures for your paper.

In [ ]:
for case_name in df['case_name'].unique():
    print(f"\n--- {case_name} ---")
    display(df[df['case_name'] == case_name].head(3)[['filename', 'class_name', 'has_gt_mask', 'dice', 'iou', 'gt_label', 'pred_label', 'pred_prob']])


## 9. (Optional) Per-class breakdown

Shows which class (e.g. Z-line, Esophagitis) drives most classification
errors — useful for discussing which structures the model confuses with
polyps.

In [24]:
crosstab
df, summary_df, crosstab, _model, _clf = run_experiment(
    class_sources=CLASS_SOURCES,
    dataset_name=DATASET_NAME,
    num_samples=NUM_SAMPLES,
    samples_per_class=SAMPLES_PER_CLASS,
    seg_threshold=SEG_THRESHOLD,
    mask_suffix=MASK_SUFFIX,
    output_dir=OUTPUT_DIR,
    save_visuals=SAVE_VISUALS,
    seed=SEED,
    model_path=MODEL_PATH,
    classifier_path=CLASSIFIER_PATH,
)
summary_df


🔄 Loading Y-Net backbone: /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/thesis_v3_sequential_v_eval_data_driven_v2.keras
🔄 Loading classifier artifact: /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/elm_artifacts.npz
⚠️ Artifact is missing ['decision_threshold', 'target_layer_name', 'img_size'] — using defaults (threshold=0.5, target_layer_name='clf_pcc_layer', img_size=256). If your feature layer or resolution differs, pass them in manually.
⚠️ Artifact has no saved val_min_score/val_max_score — falling back to logistic squashing of the raw ensemble score instead of the original min-max rescale. Threshold defaults to 0.5 (raw score 0) unless overridden.
   -> Detected classifier type: ensemble_elm | feature layer: clf_pcc_layer | threshold: 0.500
📊 Gathering image/mask pairs per class...
  [Polyp] found 15 images, using 15 (label=1, masks=yes)
  [Z-line] found 0 images, using 0 (label=0, masks=virtual-empty)
  [Esophag

,Case,Count,Percentage,Mean Dice,Mean IoU,Mean Clf Prob
0,Correct Seg + Correct Clf,12,80.00%,0.8179,0.6987,0.6902
1,Correct Seg + Incorrect Clf,2,13.33%,0.7824,0.6435,0.4469
2,Incorrect Seg + Correct Clf,1,6.67%,0.4273,0.2719,0.5135
3,Incorrect Seg + Incorrect Clf,0,0.00%,-,-,-
